# 01 — Data and exploratory analysisThe first uncomfortable fact: the target is nearly balanced and nearlyunpredictable.All pipeline logic lives in `src/`. Notebooks explore; they never define it.

In [ ]:
import sys, warningssys.path.insert(0, "../src")warnings.filterwarnings("ignore")import numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom stock_movement.config import load_configfrom stock_movement.dataset import build_datasetconfig = load_config("../configs/reproduction/readme_aapl_2026_07.yaml")dataset = build_dataset(config)dataset.summary()

## Price history and the return distribution

In [ ]:
prices = dataset.pricesfig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)axes[0].plot(prices.index, prices["Close"], lw=1)axes[0].set_yscale("log")axes[0].set_title(f"{config.data.ticker} adjusted close (log scale)")axes[0].grid(alpha=0.3)overnight = prices["Open"] / prices["Close"].shift(1) - 1intraday = prices["Close"] / prices["Open"] - 1axes[1].plot(prices.index, intraday * 100, lw=0.5, label="open-to-close (what we predict)")axes[1].plot(prices.index, overnight * 100, lw=0.5, alpha=0.6, label="overnight gap")axes[1].set_title("Session returns (%)")axes[1].legend(fontsize=8)axes[1].grid(alpha=0.3)plt.tight_layout()

## The target is the intraday move, not the whole dayThe default target is `Close(t+1) / Open(t+1) - 1`. The overnight gap is excludedbecause a strategy that enters at the open cannot capture it — that return hasalready happened by the time you can trade.

In [ ]:
comparison = pd.DataFrame({    "open_to_close": intraday,    "close_to_close": prices["Close"].pct_change(),    "overnight_gap": overnight,}).dropna()print(comparison.describe().round(5).to_string())print()print("share of close-to-close variance that is overnight:")print(f"  {comparison['overnight_gap'].var() / comparison['close_to_close'].var():.1%}")

## Volatility clusters; direction does notVolatility is strongly autocorrelated. The *sign* of the return is barelyautocorrelated at all. That asymmetry is the whole difficulty of this project.

In [ ]:
returns = comparison["open_to_close"]fig, axes = plt.subplots(1, 2, figsize=(13, 4))pd.Series([returns.autocorr(lag) for lag in range(1, 31)]).plot(kind="bar", ax=axes[0], color="steelblue")axes[0].set_title("Autocorrelation of open-to-close returns")axes[0].axhline(0, color="black", lw=1)pd.Series([returns.abs().autocorr(lag) for lag in range(1, 31)]).plot(kind="bar", ax=axes[1], color="indianred")axes[1].set_title("Autocorrelation of |returns| (volatility clustering)")axes[1].axhline(0, color="black", lw=1)plt.tight_layout()

## Class balance, overall and per year

In [ ]:
from stock_movement import plotsup_rate = dataset.y.mean()print(f"up sessions: {up_rate:.4f}   down: {1 - up_rate:.4f}")print(f"\n'always up' scores {max(up_rate, 1 - up_rate):.4f} plain accuracy")print("and exactly 0.5000 balanced accuracy. Hence balanced accuracy as the primary metric.")plots.plot_class_balance(dataset.y, pathlib.Path("/tmp/_class_balance.png"))from IPython.display import ImageImage("/tmp/_class_balance.png")

## Data provenance and the partial-bar decision

In [ ]:
target_meta = dataset.metadata["target"]print("raw sha256:", target_meta["raw_sha256"])print()print(json.dumps(target_meta["partial_bar_decision"], indent=2))print()print(json.dumps(target_meta["validation"], indent=2)[:900])